In [1]:
import json
import re
import os
from pathlib import Path

SMART_RESIZE = (1932, 1092)
IMAGE_SIZE = (1920, 1080)

W_RATIO = IMAGE_SIZE[0] / SMART_RESIZE[0]
H_RATIO = IMAGE_SIZE[1] / SMART_RESIZE[1]

RESULT_FOLDER_PATH = "/Users/lockewang/FIG/WebDomainRandomizer/results_baseline"

# Load all data from uitars_predictions and uitars_metrics

In [2]:
import pandas as pd

def load_metrics_dict(metrics_file):
    """Load all metrics from uitars_metrics.jsonl into a dictionary keyed by (episode, step_index)"""
    metrics_dict = {}
    if not metrics_file.exists():
        return metrics_dict
    
    try:
        with open(metrics_file, 'r') as f:
            for line in f:
                data = json.loads(line)
                episode = data.get('episode')
                step_index = data.get('step_index')
                if episode is not None and step_index is not None:
                    key = (episode, step_index)
                    metrics_dict[key] = data.get('metrics', {})
    except Exception as e:
        print(f"Warning: Error reading metrics file {metrics_file}: {e}")
    
    return metrics_dict

def load_data_from_results_folder(result_folder_path):
    """Load all data from uitars_predictions jsonl files and match with metrics"""
    rows = []
    result_path = Path(result_folder_path)
    
    for model_config_folder in result_path.iterdir():
        if not model_config_folder.is_dir():
            continue
        if model_config_folder.name == 'locke_qwen_thought_relational_query_2':
            continue

        print(f"Processing model config folder: {model_config_folder.name}")
        
        for run_folder in model_config_folder.iterdir():
            if not run_folder.is_dir():
                continue
            
            if not run_folder.name.startswith('run'):
                continue

            model = model_config_folder.name.split('_')[1]
            reasoning_type = "no_reasoning" if "NO" in model_config_folder.name else "reasoning"
            query_type = "direct_query" if "direct_query" in model_config_folder.name else "relational_query"
            test_split = run_folder.name.split('_')[4]
            variant = "_".join(run_folder.name.split('_')[5:])
            
            predictions_folder = run_folder / "uitars_predictions"
            metrics_file = run_folder / "uitars_metrics.jsonl"
            
            if not predictions_folder.exists():
                continue
            
            # Load all metrics for this run_folder into a dictionary
            metrics_dict = load_metrics_dict(metrics_file)
            
            # Process each task_id jsonl file in uitars_predictions
            for prediction_file in predictions_folder.glob("*.jsonl"):
                task_id = prediction_file.stem  # Get task_id from filename (without .jsonl extension)
                
                try:
                    with open(prediction_file, 'r') as f:
                        for line in f:
                            data = json.loads(line)
                            
                            episode = data.get('episode', task_id)
                            step_index = data.get('step_index')
                            instruction = data.get('instruction')
                            prediction = data.get('prediction')
                            
                            # Get ground_truth_bbox from metadata
                            metadata = data.get('metadata', {})
                            gt_bbox = metadata.get('bounding_box')
                            
                            # Get metrics from metrics_dict
                            step_metrics = metrics_dict.get((episode, step_index), {})
                            
                            row = {
                                'model': model,
                                'reasoning_type': reasoning_type,
                                'query_type': query_type,
                                'test_split': test_split,
                                'variant': variant,
                                'task_id': episode,
                                'step_index': step_index,
                                'instruction': instruction,
                                'gt_bbox': gt_bbox,
                                'prediction': prediction,
                                'action_str_em': step_metrics.get('action_str_em'),
                                'hit_box_accuracy': step_metrics.get('hit_box_accuracy'),
                                'bbox_center_mse': step_metrics.get('bbox_center_mse')
                            }
                            
                            rows.append(row)
                            
                except Exception as e:
                    print(f"Warning: Error reading prediction file {prediction_file}: {e}")
                    continue
    
    return pd.DataFrame(rows)

# Load all data
df = load_data_from_results_folder(RESULT_FOLDER_PATH)

print(f"Total rows loaded: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Processing model config folder: locke_qwen_NO_thought_relational_query
Processing model config folder: locke_qwen_thought_relational_query
Processing model config folder: locke_gta1_NO_thought_direct_query
Processing model config folder: locke_uitars_NO_thought_relational_query
Processing model config folder: locke_uitars_thought_direct_query
Processing model config folder: locke_uitars_thought_relational_query
Processing model config folder: locke_uitars_NO_thought_direct_query
Processing model config folder: locke_qwen_thought_direct_query
Processing model config folder: locke_gta1_NO_thought_relational_query
Processing model config folder: locke_qwen_NO_thought_direct_query
Total rows loaded: 11835

Columns: ['model', 'reasoning_type', 'query_type', 'test_split', 'variant', 'task_id', 'step_index', 'instruction', 'gt_bbox', 'prediction', 'action_str_em', 'hit_box_accuracy', 'bbox_center_mse']

First few rows:


,model,reasoning_type,query_type,test_split,variant,task_id,step_index,instruction,gt_bbox,prediction,action_str_em,hit_box_accuracy,bbox_center_mse
0,qwen,no_reasoning,relational_query,task,text_shrink,8dcf6423-262a-439b-9ee7-279a920468fa,0,Click on the button to the left of 'EXPERIENCE',"[370, 61, 111.265625, 55]","Action: click(start_box='(553,86)')",1.0,0.0,8114.325226
1,qwen,no_reasoning,relational_query,task,text_shrink,e6f6e6c8-f1e6-42bb-a3af-696ed8de571b,0,Click on the link above 'Flight status',"[607.046875, 174.390625, 65.390625, 12]","Action: click(start_box='(640,211)')",1.0,0.0,468.500153
2,qwen,no_reasoning,relational_query,task,text_shrink,400c291f-6a0c-46fb-874e-d5c174fdedfc,0,Click on the button below 'Infant +' button,"[895.5, 533.5, 32.5, 13]","Action: click(start_box='(834,500)')",1.0,0.0,3822.531250
3,qwen,no_reasoning,relational_query,task,text_shrink,5b888855-b921-4c61-8f79-73902ee0eafa,0,Type the textbox above '* PIN Number',"[448, 461.953125, 380, 36.890625]","Action: click(start_box='(641,563)')",0.0,0.0,3416.009064
4,qwen,no_reasoning,relational_query,task,text_shrink,9ceab2a3-7919-4f15-871a-21638fd93b24,0,Select '11:00 AM' in the select to the left of...,"[1036.40625, 202, 112, 62]","Action: click(start_box='(1356,247)')",0.0,0.0,34838.832520


# For each model_config, check how many types of predictions are there

In [28]:
def categorize_prediction_pattern(text):
    """Categorize prediction text into pattern types"""
    if pd.isna(text):
        return 'missing'
    
    text_str = str(text)
    
    # Pattern to match: Action: click(start_box='(number, number)')
    normal_pattern = r"^Action:\s*click\(start_box='\(\d+,\d+\)'\)$"
    
    types = []

    # Check if contains 'Thought' (case-insensitive)
    if 'Thought' in text_str or 'thought' in text_str:
        types.append('contains_thought')
    
    # Check if contains 'tool_call' (case-insensitive)
    if 'tool_call' in text_str or 'toolCall' in text_str or 'tool-call' in text_str:
        types.append('contains_tool_call')
    
    # Check if has exactly 4 numbers (could be bbox format like [x1, y1, x2, y2] or (x1, y1, x2, y2))
    numbers = re.findall(r'\d+', text_str)
    if len(numbers) == 4:
        types.append('four_numbers')
    
    # # Check for other bbox formats like [x1,y1,x2,y2], Box(x1,y1,x2,y2), _bbox(...), etc.
    # bbox_patterns = [
    #     r'\[[\d\s,]+\]',  # [numbers]
    #     r'Box\([^)]+\)',  # Box(...)
    #     r'_bbox\([^)]+\)',  # _bbox(...)
    #     r'bbox\([^)]+\)',  # bbox(...)
    # ]
    # for pattern in bbox_patterns:
    #     if re.search(pattern, text_str):
    #         return 'bbox_format'
    
    # Check for other action types (scroll, type, etc.)
    if re.search(r'Action:\s*(scroll|type|select|wait|finished)', text_str, re.IGNORECASE):
        action_type = re.search(r'Action:\s*(scroll|type|select|wait|finished)', text_str, re.IGNORECASE).group(1)
        types.append(f'{action_type}')

    # Check for normal format (exact match)
    if re.match(normal_pattern, text_str.strip()):
        types.append('simple_click')
    
    # Everything else is unexpected
    if len(types) == 0:
        types.append('other_unexpected')

    return ' + '.join(types)

In [ ]:
pd.set_option('display.max_rows', None)

df['prediction_type'] = df['prediction'].apply(categorize_prediction_pattern)

other_unexpected_count = df[df['prediction_type'] == 'other_unexpected'].sum()
other_unexpected_count


model                 0
reasoning_type        0
query_type            0
test_split            0
variant               0
task_id               0
step_index            0
instruction           0
gt_bbox               0
prediction            0
action_str_em       0.0
hit_box_accuracy    0.0
bbox_center_mse     0.0
prediction_type       0
dtype: object

In [36]:
for model in df['model'].unique():
    result = df[df['model'] == model].groupby(['model', 'reasoning_type', 'query_type', 'variant', 'prediction_type'])['prediction'].count()
    styled = (result.reset_index(name='count')
              .style
              .background_gradient(subset='count')
              .format({'count': '{:,}'}))
    print("="*80)
    print(model)
    display(styled)
    print("="*80)


qwen


,model,reasoning_type,query_type,variant,prediction_type,count
0,qwen,no_reasoning,direct_query,original,simple_click,298
1,qwen,no_reasoning,direct_query,precision,simple_click,298
2,qwen,no_reasoning,direct_query,style,simple_click,293
3,qwen,no_reasoning,direct_query,text_shrink,contains_tool_call,1
4,qwen,no_reasoning,direct_query,text_shrink,simple_click,294
5,qwen,no_reasoning,relational_query,original,contains_tool_call,1
6,qwen,no_reasoning,relational_query,original,simple_click,297
7,qwen,no_reasoning,relational_query,precision,simple_click,298
8,qwen,no_reasoning,relational_query,style,contains_tool_call,1
9,qwen,no_reasoning,relational_query,style,simple_click,292


gta1


,model,reasoning_type,query_type,variant,prediction_type,count
0,gta1,no_reasoning,direct_query,original,simple_click,298
1,gta1,no_reasoning,direct_query,precision,simple_click,298
2,gta1,no_reasoning,direct_query,style,simple_click,293
3,gta1,no_reasoning,direct_query,text_shrink,simple_click,295
4,gta1,no_reasoning,relational_query,original,simple_click,298
5,gta1,no_reasoning,relational_query,precision,simple_click,298
6,gta1,no_reasoning,relational_query,style,simple_click,293
7,gta1,no_reasoning,relational_query,text_shrink,simple_click,295


uitars


,model,reasoning_type,query_type,variant,prediction_type,count
0,uitars,no_reasoning,direct_query,original,scroll,12
1,uitars,no_reasoning,direct_query,original,simple_click,286
2,uitars,no_reasoning,direct_query,precision,scroll,4
3,uitars,no_reasoning,direct_query,precision,simple_click,294
4,uitars,no_reasoning,direct_query,style,scroll,10
5,uitars,no_reasoning,direct_query,style,simple_click,283
6,uitars,no_reasoning,direct_query,text_shrink,scroll,10
7,uitars,no_reasoning,direct_query,text_shrink,simple_click,285
8,uitars,no_reasoning,relational_query,original,scroll,27
9,uitars,no_reasoning,relational_query,original,simple_click,271
